In [2]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from skimage import measure
from cellpose import models
import import_images  # seu script de carregar imagens

# Bloqueia GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Caminho da pasta com imagens
base_dir = "/home/kayllany.oliveira/remote-repos/CellViability/data/Teste"

# Encontra todas as imagens .tif
image_paths = import_images.encontrar_imagens_tiff(base_dir)

# Cria pasta de saída
output_dir = os.path.join(os.getcwd(), "resultados", "cellpose_sam")
os.makedirs(output_dir, exist_ok=True)

# Inicializa modelo Cellpose-SAM
modelo = models.CellposeModel(
    gpu=False,
    pretrained_model='cpsam',
   
)

# Loop por cada imagem
for idx, img_path in image_paths.items():
    print(f"Processando {img_path} ...")
    
    # Carrega imagem usando seu script
    img = import_images.carregar_imagem_por_indice(image_paths, idx)
    if img is None:
        print("Falha ao carregar imagem, pulando...")
        continue
    
    # Avaliação (grayscale, sem rescale)
    masks, flows, styles = modelo.eval(
        [img],
        normalize=True,
        #batch_size=1,
        resample=True,
        channels=[0, 0],
        diameter=15,
        flow_threshold=1,
        cellprob_threshold=1.5,
        min_size=30,
        max_size_fraction=50,
        compute_masks=True,
        interp=True
    )

    # Nome original da imagem
    nome_original = os.path.splitext(os.path.basename(img_path))[0]

    # ---------- Salva máscara preenchida com fundo transparente ----------
    mask_array = np.zeros((*masks[0].shape, 4), dtype=np.uint8)  # RGBA
    mask_array[masks[0] > 0] = [0, 255, 0, 255]  # células verdes
    mask_path = os.path.join(output_dir, f"{nome_original}_mask.png")
    Image.fromarray(mask_array).save(mask_path)
    print(f"Máscara salva em: {mask_path}")

    # ---------- Cria overlay apenas com contornos ----------
    fig, ax = plt.subplots()
    ax.imshow(img, cmap='gray')
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)  # remove margens

    # percorre cada célula
    for cell_label in np.unique(masks[0]):
        if cell_label == 0:
            continue
        cell_mask = masks[0] == cell_label
        contours = measure.find_contours(cell_mask, 0.5)
        for contour in contours:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=1, color='lime')  # contorno verde

    ax.axis('off')
    overlay_path = os.path.join(output_dir, f"{nome_original}_overlay_contorno.png")
    plt.savefig(overlay_path, dpi=600, transparent=True)
    plt.close(fig)
    print(f"Overlay com contorno salvo em: {overlay_path}")

print("Processamento concluído para todas as imagens.")



pretrained_model path does not exist, using default model


/home/kayllany.oliveira/remote-repos/CellViability/data/Teste/001002-1-001001001.tif
/home/kayllany.oliveira/remote-repos/CellViability/data/Teste/007019-1-001001001.tif
/home/kayllany.oliveira/remote-repos/CellViability/data/Teste/001024-1-001001001.tif
Processando /home/kayllany.oliveira/remote-repos/CellViability/data/Teste/001002-1-001001001.tif ...
Imagem carregada: /home/kayllany.oliveira/remote-repos/CellViability/data/Teste/001002-1-001001001.tif - Dimensão: (1024, 1360)
Máscara salva em: /home/kayllany.oliveira/remote-repos/CellViability/resultados/cellpose_sam/001002-1-001001001_mask.png
Overlay com contorno salvo em: /home/kayllany.oliveira/remote-repos/CellViability/resultados/cellpose_sam/001002-1-001001001_overlay_contorno.png
Processando /home/kayllany.oliveira/remote-repos/CellViability/data/Teste/001024-1-001001001.tif ...
Imagem carregada: /home/kayllany.oliveira/remote-repos/CellViability/data/Teste/001024-1-001001001.tif - Dimensão: (1024, 1360)
Máscara salva em: /h